## Prediction for Step 6 (from reading the scripts, before running)
- `astral` may not be installed (not in the documented dependencies).
- config.py paths are placeholders ("path/to/..."), so they must be overridden for the test.
- weather_match joins on "Datetime" but the weather table's column is "datetime": expect KeyError.
- If worked around: weather has only the 16:00 and 17:00 hours; the matcher rounds to the NEAREST hour, so 1,148 clips match 17:00 and 599 clips (17:30 onward) round to 18:00 and get blank weather.
- Step 6 ignores run_pipeline's folders and does not write outputs/weather.csv.
- Run as-is: weather_match() should raise KeyError: 'Datetime' (the weather table has 'datetime').
- After working around that: 1,148 clips match 17:00; 599 clips round to 18:00 and get blank weather; the 16:00 row is never used.

In [1]:
import sys, importlib.metadata as md
from pathlib import Path
import pandas as pd

HOME = Path.home()
RAW_DIR      = HOME / "kwf_test" / "raw_audio"
CLIPS_DIR    = HOME / "kwf_test" / "split_clips"
OUTPUTS_DIR  = HOME / "kwf_test" / "outputs"
PIPELINE_DIR = Path("/Users/huyguy/Documents/Documents - Huys MacBook Pro/Job Materials/KwF/Inference Pipeline/KWF_Human_Presence/inference_pipeline")
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

try:
    print("astral", md.version("astral"))
except md.PackageNotFoundError:
    print("astral NOT INSTALLED")

import config
for name in ["AUDIO_MOTH_FOLDER", "WEATHER_DATA_PATH", "WEATHER_OUTPUT_CSV",
             "METADATA_OUTPUT_CSV", "LATITUDE", "LONGITUDE", "TIMEZONE"]:
    print(f"{name:22s} {getattr(config, name)}")

try:
    import weather_info, weather_matching
    print("\nImports OK")
except Exception as e:
    print("\nIMPORT PROBLEM:", type(e).__name__, e)

astral NOT INSTALLED
AUDIO_MOTH_FOLDER      path/to/raw
WEATHER_DATA_PATH      path/to/weather_data.csv
WEATHER_OUTPUT_CSV     path/to/weather_matches.csv
METADATA_OUTPUT_CSV    path/to/metadata.csv
LATITUDE               8.6602
LONGITUDE              -83.6502
TIMEZONE               America/Costa_Rica

IMPORT PROBLEM: ModuleNotFoundError No module named 'astral'


What this does: checks that astral is installed, prints the config.py values Step 6 depends on, and tries to import both weather scripts.

How to read the output: “NOT INSTALLED” for astral means the import will fail and the pipeline would crash on a fresh machine. The config lines show which values are still placeholders (path/to/...). “Imports OK” means the scripts load, and any error shows exactly what’s missing.

In [2]:
%pip install astral

Note: you may need to restart the kernel to use updated packages.


In [3]:
import importlib.metadata as md
print("astral", md.version("astral"))
try:
    import weather_info, weather_matching
    print("Imports OK")
except Exception as e:
    print("IMPORT PROBLEM:", type(e).__name__, e)

astral 3.2
Imports OK


In [4]:
import time

weather_csv = OUTPUTS_DIR / "nb06_weather_data.csv"

# Override the placeholder paths inside the module (repo files are not changed)
weather_info.AUDIO_MOTH_FOLDER = str(RAW_DIR)
weather_info.WEATHER_DATA_PATH = str(weather_csv)

t0 = time.time()
weather_info.get_weather_info_all()
print(f"Runtime: {time.time() - t0:.1f}s")

if weather_csv.exists():
    w = pd.read_csv(weather_csv)
    print("\nShape:", w.shape)
    print("Columns:", w.columns.tolist())
    print(w[["datetime", "temperature", "precipitation", "humidity", "weather_desc", "time_of_day"]].to_string(index=False))
else:
    print("\nNo weather file was written.")

Processing: 20250319_163237.WAV
[OK] Combined weather data saved to /Users/huyguy/kwf_test/outputs/nb06_weather_data.csv
Runtime: 1.0s

Shape: (2, 11)
Columns: ['datetime', 'temperature', 'windspeed', 'precipitation', 'humidity', 'weathercode', 'weather_desc', 'timestamp_local', 'sunrise', 'sunset', 'time_of_day']
           datetime  temperature  precipitation  humidity weather_desc time_of_day
2025-03-19 16:00:00         27.8            0.0        76    Clear sky     evening
2025-03-19 17:00:00         27.1            0.0        80    Clear sky     evening


What this does: points weather_info’s paths at the test folders, runs get_weather_info_all() (the same call run_pipeline.py makes), and loads the weather table it writes.

How to read the output: “Processing: 20250319_163237.WAV” means it found the raw recording. “[OK] Combined weather data saved” means the download worked. An [ERROR] line or “No weather data collected” means no table was made, and the printed error is the finding. The table summary show the hours the recording covers (16:00 and 17:00 on 2025-03-19). The columns show what Step 7 would get.

In [5]:
import config, traceback

config.METADATA_OUTPUT_CSV = str(OUTPUTS_DIR / "nb05_metadata.csv")
config.WEATHER_DATA_PATH   = str(OUTPUTS_DIR / "nb06_weather_data.csv")
config.WEATHER_OUTPUT_CSV  = str(OUTPUTS_DIR / "nb06_weather_matches.csv")

try:
    weather_matching.weather_match()
    print("weather_match ran without an error")
except Exception as e:
    print("weather_match FAILED:", type(e).__name__, e)

weather_match FAILED: KeyError 'Datetime'


What this does: points config’s paths at the test files (metadata from Step 5, weather from the download), then calls weather_match() exactly as run_pipeline.py does. The call is wrapped in try/except, so a crash is recorded, not fatal.

How to read the output: “KeyError: ‘Datetime’” confirms the column-name mismatch, and Step 6 can’t finish as written.

In [6]:
w = pd.read_csv(OUTPUTS_DIR / "nb06_weather_data.csv")
w = w.rename(columns={"datetime": "Datetime"})
w.to_csv(OUTPUTS_DIR / "nb06_weather_data_renamed.csv", index=False)

config.WEATHER_DATA_PATH = str(OUTPUTS_DIR / "nb06_weather_data_renamed.csv")

try:
    weather_matching.weather_match()
    print("weather_match ran without an error")
except Exception as e:
    print("weather_match FAILED:", type(e).__name__, e)

weather_match ran without an error


What this does: makes a copy of the weather table with the time column renamed from datetime to Datetime, points config at the copy, and reruns weather_match().

How to read the output: “ran without an error” means the workaround got past the name mismatch. This is a test workaround, not a fix, because the repo scripts are unchanged.

In [7]:
mm = pd.read_csv(OUTPUTS_DIR / "nb06_weather_matches.csv")
print("Shape:", mm.shape)
print("Columns:", mm.columns.tolist())
print("Unique clip_names:", mm["clip_name"].nunique())

has_weather = mm["temperature"].notna()
print("\nClips with weather:", has_weather.sum(), "| clips with blank weather:", (~has_weather).sum())
print("Matched hour counts:\n", mm.loc[has_weather, "Datetime"].value_counts().to_string())

ts = pd.to_datetime(mm["clip_name"].str.extract(r"(\d{8}_\d{6})")[0], format="%Y%m%d_%H%M%S")
print("\nFirst clip WITHOUT weather:", ts[~has_weather].min())
print("Last clip WITH weather:   ", ts[has_weather].max())

Shape: (1747, 12)
Columns: ['Datetime', 'temperature', 'windspeed', 'precipitation', 'humidity', 'weathercode', 'weather_desc', 'timestamp_local', 'sunrise', 'sunset', 'time_of_day', 'clip_name']
Unique clip_names: 1747

Clips with weather: 1148 | clips with blank weather: 599
Matched hour counts:
 Datetime
2025-03-19 17:00:00    1148

First clip WITHOUT weather: 2025-03-19 17:30:01
Last clip WITH weather:    2025-03-19 17:29:58


What this does: loads the matched table and checks it against the prediction: row count, which clips got weather, and which hour each clip was matched to.

How to read the output: I predicted 1,747 rows, with 1,148 clips matched to the 17:00 weather and 599 clips with blank weather, because their times round to 18:00, an hour that isn’t in the table. The blank rows are the key check. A merge that can’t find a match leaves blank values without raising any error, so the only way to notice is to count them.

**Finding 1 (confirmed):** `weather_match()` fails with `KeyError: 'Datetime'`. `weather_matching.py` merges on "Datetime" but `weather_info.py` saves the column as "datetime". Step 6 cannot finish as written, so `run_pipeline.py` would crash here.

In [9]:
import os, tempfile, shutil

test_root = Path(tempfile.mkdtemp(prefix="lower_wav_test_"))
day = test_root / "Audio_Moth_5_031925"
day.mkdir()
os.symlink(RAW_DIR / "Audio_Moth_5_031925" / "20250319_163237.WAV", day / "20250319_163237.wav")
print("Files:", [p.name for p in day.iterdir()])

orig = (weather_info.AUDIO_MOTH_FOLDER, weather_info.WEATHER_DATA_PATH)
out = test_root / "weather_lower.csv"
weather_info.AUDIO_MOTH_FOLDER = str(test_root)
weather_info.WEATHER_DATA_PATH = str(out)
try:
    weather_info.get_weather_info_all()
    print("weather file written:", out.exists())
finally:
    weather_info.AUDIO_MOTH_FOLDER, weather_info.WEATHER_DATA_PATH = orig
    shutil.rmtree(test_root)

Files: ['20250319_163237.wav']
[WARNING] No weather data collected.
weather file written: False


What this does: builds a temporary raw folder holding a lowercase-named link to the recorder 5 recording, points weather_info at it, and runs get_weather_info_all(). It restores the module’s original paths afterward.

How to read the output: “No weather data collected” and “weather file written: False” confirm that lowercase .wav raw files are silently ignored.

## Observations (Step 6)

**Tested:** `weather_info.get_weather_info_all()` and `weather_matching.weather_match()` on the recorder 5 recording and its 1,747 clips, with config paths overridden in the notebook.

**Download (works):** 1 raw file in, 2 hourly rows out (2025-03-19 16:00 and 17:00; 11 columns), 1.0 s. Location is one fixed point from `config.py` for all recorders.

**Matching:** 1,747 rows out, 1 per clip, but only 1,148 clips (34% missing for 599) have weather. All 1,148 matched 17:00. The 599 clips from 17:30:01 onward round to 18:00, which is not in the weather table, so their weather is blank with no error.

**Issues**
1. `weather_match()` fails with `KeyError: 'Datetime'`: `weather_matching.py` merges on "Datetime" but `weather_info.py` saves "datetime". Step 6 cannot finish as written. (Workaround used for the rest of the test: renamed copy of the weather table; repo files unchanged.)
2. Hour mismatch: the download keeps hours counted from the start time, while the matcher rounds to the nearest hour, so clips in the second half of the last hour get blank weather. Intended rule (nearest hour or hour the clip falls in)?
3. `astral` was not installed and is not in the documented dependencies (`run_pipeline.py` imports `weather_info` at startup). Installed astral 3.2.
4. All file paths in `config.py` are placeholders (`path/to/...`). Step 6 takes no arguments, ignores `--raw_audio`, reads the raw recordings folder and the Step 5 metadata path from `config.py`, and does not write the `outputs/weather.csv` that `run_pipeline.py` returns.
5. `time_of_day` uses fixed clock-hour ranges (16:00 to 19:00 = "evening"), not sunrise and sunset.

**Robustness note (low risk):** the download only finds raw files ending in uppercase ".WAV" (hardcoded `glob("*.WAV")`). A lowercase ".wav" raw file was ignored with "No weather data collected" and no error. Step 5 has the opposite rule (lowercase only), so the two steps disagree on extension case.

**Not tested:** Step 6 through `run_pipeline.py`; multiple recordings or recorders (several in the same hour could duplicate rows); recent dates (the archive service may lag); behavior without internet.